# AMAC-ECG — Stage 2B: CNN training on Google Colab

This notebook clones the **public AMAC-ECG GitHub repository**, uploads the corrected **DS1 dataset only**, verifies that it exactly matches the frozen `src/folds.json`, and runs the CNN grid on a Colab GPU.

### Protocol safeguards
- Uses DS1 only for training and patient-grouped cross-validation.
- Does **not** load or evaluate on DS2. DS2 was accessed in an earlier project stage, so it must not be described as an untouched test set.
- Uses the committed `src/folds.json`; it does not regenerate folds.
- Preserves the existing LR/XGBoost summary before the CNN script writes its own summary.
- Keeps existing LR/XGBoost prediction files untouched.
- Default experiment: 5 strategies × 2 RR settings × 5 folds = **50 CNN fold-runs**.

**Before running:** In Colab select **Runtime → Change runtime type → T4 GPU**. Run cells from top to bottom. Do not run the CNN training cell twice manually; the script supports resuming valid completed fold predictions if Colab disconnects.

## 1. Clone the GitHub repository

This retrieves the code and frozen folds. The dataset files are intentionally excluded from GitHub.

In [1]:
import os, shutil, subprocess, sys

REPO_URL = "https://github.com/sanjanapriyadarsini/AMAC-ECG.git"
WORK = "/content/amac_ecg"
PROJECT = os.path.join(WORK, "AMAC-ECG")

if os.path.exists(WORK):
    shutil.rmtree(WORK)
os.makedirs(WORK, exist_ok=True)

subprocess.run(["git", "clone", REPO_URL, PROJECT], check=True)
os.chdir(PROJECT)

print("Project:", PROJECT)
print("Top-level files:", sorted(os.listdir(".")))
assert os.path.isfile("src/run_grid.py")
assert os.path.isfile("src/folds.json")
assert os.path.isfile("artifacts/grid_summary.json")

Project: /content/amac_ecg/AMAC-ECG
Top-level files: ['.git', '.gitignore', 'DATA_README.txt', 'IMPORTANT_BEFORE_RUNNING.txt', 'RUN_ORDER_CORRECTED.md', 'artifacts', 'requirements.txt', 'src', 'stage1.md', 'stage2.md']


## 2. Select and verify the GPU

If this cell says CUDA is unavailable, stop and change the Colab runtime to T4 GPU before continuing.

In [2]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("GPU not detected. In Colab choose Runtime → Change runtime type → T4 GPU.")
print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.14.1+cu130
CUDA available: True
GPU: Tesla T4


## 3. Upload the corrected DS1 file

On your computer, select **only** `artifacts/ds1.npz` from your local `ECG_Project` folder.

Do **not** upload `ds2.npz`, the whole project ZIP, or the predictions folder. The DS1 file is approximately 12.23 MB.

In [3]:
from google.colab import files
import os

uploaded = files.upload()
uploaded_npz = [name for name in uploaded if name.lower().endswith(".npz")]
if len(uploaded_npz) != 1:
    raise RuntimeError(f"Please upload exactly one .npz file (DS1 only). Received: {uploaded_npz}")

os.makedirs("artifacts", exist_ok=True)
source_name = uploaded_npz[0]
target_path = os.path.join("artifacts", "ds1.npz")
if os.path.abspath(source_name) != os.path.abspath(target_path):
    os.replace(source_name, target_path)

print("DS1 placed at:", target_path)
print("Size (MB):", round(os.path.getsize(target_path) / 1e6, 2))

Saving ds1.npz to ds1.npz
DS1 placed at: artifacts/ds1.npz
Size (MB): 12.83


## 4. Install project dependencies

In [4]:
%pip -q install -r requirements.txt
print("Dependencies installed.")

Dependencies installed.


## 5. Verify DS1, SHA-256, and frozen patient folds

This is a hard gate. Training will not start unless the uploaded DS1 matches the exact hash recorded in the repository's `src/folds.json` and all expected shapes/patient IDs match.

In [5]:
import hashlib, json, numpy as np, os

ds1_path = "artifacts/ds1.npz"
folds_path = "src/folds.json"
assert os.path.isfile(ds1_path), "artifacts/ds1.npz is missing."
assert os.path.isfile(folds_path), "src/folds.json is missing."

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

dataset_hash = sha256_file(ds1_path)
with np.load(ds1_path) as d:
    required = {"X", "rr", "y", "pid", "record_id"}
    missing = required - set(d.files)
    assert not missing, f"Missing DS1 arrays: {missing}"
    X, rr, y, pid = d["X"], d["rr"], d["y"], d["pid"]
    record_id = d["record_id"]
    assert X.shape == (53111, 216), f"Unexpected X shape: {X.shape}"
    assert rr.shape == (53111, 4), f"Unexpected RR shape: {rr.shape}"
    assert y.shape == pid.shape == record_id.shape == (53111,)
    assert np.isfinite(X).all() and np.isfinite(rr).all()
    n_patients = len(np.unique(pid))

with open(folds_path) as f:
    fold_spec = json.load(f)

assert fold_spec["n_beats"] == len(y), "Fold file beat count does not match DS1."
assert fold_spec["ds1_sha256"] == dataset_hash, "HASH MISMATCH: folds.json was generated from a different DS1 file."
assert set(fold_spec["patient_fold"]) == {str(int(p)) for p in np.unique(pid)}, "Patient IDs in folds do not match DS1."
assert n_patients == 22, f"Unexpected DS1 patient count: {n_patients}"
assert fold_spec["n_folds"] == 5

print("PASS — DS1 SHA-256:", dataset_hash)
print("PASS — X:", X.shape, "| RR:", rr.shape)
print("PASS — beats:", len(y), "| patients:", n_patients)
print("PASS — frozen 5-fold patient assignment matches this exact DS1.")
print("DS2 has not been loaded or used in this notebook.")

PASS — DS1 SHA-256: 74bfdda3fe8d2d73b13b00596271eb15418ee1bd13046e0aa7e42186eba54644
PASS — X: (53111, 216) | RR: (53111, 4)
PASS — beats: 53111 | patients: 22
PASS — frozen 5-fold patient assignment matches this exact DS1.
DS2 has not been loaded or used in this notebook.


## 6. Preserve the baseline summary, then run the CNN grid

The existing `artifacts/grid_summary.json` contains the LR/XGBoost baseline rows. `run_grid.py --models cnn` writes a CNN-only summary to that same filename, so this cell first creates a separate backup.

The command runs 50 fold-runs: 5 strategies × 2 RR settings × 5 patient-grouped folds. It uses 40 epochs maximum with the training script's early-stopping logic.

In [6]:
import os, shutil, json

existing_summary = "artifacts/grid_summary.json"
backup_summary = "artifacts/grid_summary_lr_xgb_backup.json"

assert os.path.isfile(existing_summary), "Baseline grid_summary.json is missing."
if not os.path.isfile(backup_summary):
    shutil.copy2(existing_summary, backup_summary)
    print("Saved baseline summary backup:", backup_summary)
else:
    print("Baseline summary backup already exists; leaving it unchanged.")

with open(backup_summary) as f:
    baseline_rows_before = json.load(f)
print("Baseline summary rows preserved:", len(baseline_rows_before))

# Run once. If Colab disconnects, rerun this cell: valid completed folds are cached and reused.
result = subprocess.run(
    [sys.executable, "src/run_grid.py", "--models", "cnn", "--seed", "0", "--epochs", "40"],
    text=True
)
if result.returncode != 0:
    raise RuntimeError(f"CNN grid failed with exit code {result.returncode}. Review the log above before retrying.")
print("CNN grid command finished successfully.")

Saved baseline summary backup: artifacts/grid_summary_lr_xgb_backup.json
Baseline summary rows preserved: 16
CNN grid command finished successfully.


## 7. Verify all 50 CNN fold predictions and their dataset hashes

This checks each expected file exists, contains the required prediction fields, and was produced against the exact DS1 hash verified above.

In [7]:
import os, numpy as np

strategies = ["none", "class_weight", "focal", "adasyn", "smote"]
rr_tags = ["rr", "norr"]
expected = [
    os.path.join("artifacts", "preds", f"cnn_{strategy}_{rr_tag}_f{fold}.npz")
    for strategy in strategies
    for rr_tag in rr_tags
    for fold in range(5)
]

missing = [p for p in expected if not os.path.isfile(p)]
assert not missing, f"Missing {len(missing)} CNN prediction files: {missing}"

for path in expected:
    with np.load(path, allow_pickle=False) as pred:
        required = {"proba", "y_true", "pid", "val_idx", "dataset_hash", "macro_f1"}
        absent = required - set(pred.files)
        assert not absent, f"{path} missing fields: {absent}"
        saved_hash = str(pred["dataset_hash"].item())
        assert saved_hash == dataset_hash, f"Dataset hash mismatch in {path}"

print("PASS — all", len(expected), "CNN fold-prediction files exist.")
print("PASS — every prediction file records the verified DS1 hash.")

PASS — all 50 CNN fold-prediction files exist.
PASS — every prediction file records the verified DS1 hash.


## 8. Merge CNN summary rows with the preserved LR/XGBoost baseline rows

The CNN run writes 10 configuration rows (5 strategies × 2 RR settings). This cell merges those with the original baseline summary backup. It does not retrain LR/XGBoost.

In [8]:
import json

with open("artifacts/grid_summary.json") as f:
    cnn_only_summary = json.load(f)
with open("artifacts/grid_summary_lr_xgb_backup.json") as f:
    baseline_summary = json.load(f)

cnn_rows = [row for row in cnn_only_summary if row.get("model") == "cnn"]
assert len(cnn_rows) == 10, f"Expected 10 CNN configurations; found {len(cnn_rows)}."
assert len(baseline_summary) == 16, f"Expected 16 original LR/XGBoost rows; found {len(baseline_summary)}."

merged = [row for row in baseline_summary if row.get("model") != "cnn"] + cnn_rows
with open("artifacts/grid_summary.json", "w") as f:
    json.dump(merged, f, indent=2)

print("Merged summary rows:", len(merged))
print("Preserved LR/XGBoost rows:", len(baseline_summary))
print("CNN rows:", len(cnn_rows))
for row in cnn_rows:
    print(
        f'{row["strategy"]:12} RR={int(row["use_rr"])}  '
        f'macro-F1={row["macro_f1_mean"]:.4f} ± {row["macro_f1_std"]:.4f}  '
        f'S={row["per_class_f1"]["S"]:.4f}  '
        f'V={row["per_class_f1"]["V"]:.4f}  '
        f'F={row["per_class_f1"]["F"]:.4f}'
    )

Merged summary rows: 26
Preserved LR/XGBoost rows: 16
CNN rows: 10
none         RR=1  macro-F1=0.5354 ± 0.0612  S=0.3012  V=0.7696  F=0.0218
none         RR=0  macro-F1=0.4599 ± 0.0635  S=0.1395  V=0.7571  F=0.0123
class_weight RR=1  macro-F1=0.5277 ± 0.0973  S=0.3096  V=0.7648  F=0.0668
class_weight RR=0  macro-F1=0.4190 ± 0.1186  S=0.0625  V=0.7530  F=0.0750
focal        RR=1  macro-F1=0.5470 ± 0.1302  S=0.4379  V=0.7500  F=0.0891
focal        RR=0  macro-F1=0.4135 ± 0.1057  S=0.1695  V=0.6371  F=0.0465
adasyn       RR=1  macro-F1=0.4804 ± 0.0452  S=0.2608  V=0.7513  F=0.0060
adasyn       RR=0  macro-F1=0.4181 ± 0.0322  S=0.0348  V=0.7703  F=0.0040
smote        RR=1  macro-F1=0.4966 ± 0.0525  S=0.1939  V=0.8155  F=0.0381
smote        RR=0  macro-F1=0.4072 ± 0.0541  S=0.0648  V=0.7105  F=0.0804


## 9. Package the CNN outputs for download

The output ZIP contains the 50 CNN prediction files, merged summary, baseline summary backup, frozen folds, and manifest. It deliberately excludes DS1/DS2 datasets. Keep the downloaded ZIP safe and do not upload dataset/prediction artifacts to the public GitHub repository.

In [9]:
import zipfile, os
from google.colab import files

out_zip = "/content/amac_ecg_cnn_outputs.zip"
package_files = expected + [
    "artifacts/grid_summary.json",
    "artifacts/grid_summary_lr_xgb_backup.json",
    "src/folds.json",
    "artifacts/manifest.json",
]
missing_package = [p for p in package_files if not os.path.isfile(p)]
assert not missing_package, f"Cannot package missing files: {missing_package}"

with zipfile.ZipFile(out_zip, "w", zipfile.ZIP_DEFLATED) as z:
    for path in package_files:
        z.write(path, arcname=path)

print("Created:", out_zip, "| size MB:", round(os.path.getsize(out_zip) / 1e6, 2))
files.download(out_zip)

Created: /content/amac_ecg_cnn_outputs.zip | size MB: 10.19


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Completion checklist

Do not report the CNN stage as complete unless:
- GPU was detected.
- DS1 SHA-256 matched `src/folds.json`.
- All 50 CNN fold prediction files passed hash checks.
- The merged summary contains 16 LR/XGBoost rows plus 10 CNN rows.
- The output ZIP was downloaded successfully.

This is a DS1 cross-validation experiment, not an untouched external test evaluation.